# Qwen2.5-7B Unsloth 全流程

这个 notebook 说明如何用 Unsloth 做 Qwen2.5-7B 的低显存训练、LoRA/QLoRA、保存、合并和导出。

Unsloth 的价值是把训练、加速、低显存加载和导出流程做了较多封装，适合单卡用户把训练结果继续衔接到部署。

## 1. 适合场景

- 想用更低显存训练 LoRA/QLoRA。
- 想减少手动配置 Transformers、PEFT、bitsandbytes 的细节。
- 想训练后保存 adapter、合并模型、导出 GGUF。

不适合场景：

- 需要完全手写训练循环。
- 目标模型或训练方式暂时不在 Unsloth 支持范围内。

## 2. 加载模型

关键参数：

- `model_name`：模型名。
- `max_seq_length`：最大上下文长度。
- `load_in_4bit`：是否 4bit 加载。开启后更省显存。
- `dtype`：计算精度，通常让 Unsloth 自动判断。

In [ ]:
# 安装 unsloth 后可执行。
# from unsloth import FastLanguageModel
#
# model_name = "Qwen/Qwen2.5-7B-Instruct"
# max_seq_length = 2048
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=model_name,
#     max_seq_length=max_seq_length,
#     dtype=None,
#     load_in_4bit=True,
# )

## 3. 添加 LoRA adapter

关键参数：

- `r`：LoRA 秩，越大表达能力越强，但参数更多。
- `lora_alpha`：缩放系数。
- `lora_dropout`：dropout，数据少时有助于降低过拟合。
- `target_modules`：注入哪些层。Qwen 常见是注意力投影和 MLP 投影层。

In [ ]:
# model = FastLanguageModel.get_peft_model(
#     model,
#     r=16,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     lora_alpha=32,
#     lora_dropout=0.05,
#     bias="none",
#     use_gradient_checkpointing="unsloth",
#     random_state=42,
# )

## 4. SFT 训练

Unsloth 通常和 TRL 的 `SFTTrainer` 配合。数据仍然使用本仓库的 `messages` JSONL。

In [ ]:
# from datasets import load_dataset
# from transformers import TrainingArguments
# from trl import SFTTrainer
#
# dataset = load_dataset("json", data_files="../../data/sample_sft.jsonl", split="train")
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=dataset,
#     args=TrainingArguments(
#         output_dir="../../outputs/qwen2_5_7b/unsloth_sft",
#         per_device_train_batch_size=1,
#         gradient_accumulation_steps=8,
#         learning_rate=2e-4,
#         num_train_epochs=1,
#         logging_steps=1,
#         report_to="none",
#     ),
# )
# trainer.train()

## 5. 推理模式

训练后做推理前，使用 `FastLanguageModel.for_inference(model)` 切换到推理优化模式。

In [ ]:
# FastLanguageModel.for_inference(model)
# messages = [{"role": "user", "content": "解释 QLoRA 的优点。"}]
# prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 6. 保存、合并、导出

常见产物：

- adapter：文件小，需要基座模型配合。
- merged model：合并到基座模型后的 Hugging Face 格式。
- GGUF：给 Ollama、llama.cpp 使用。

In [ ]:
# 保存 LoRA adapter
# model.save_pretrained("../../outputs/qwen2_5_7b/unsloth_adapter")
# tokenizer.save_pretrained("../../outputs/qwen2_5_7b/unsloth_adapter")
#
# 保存合并后的 16bit 模型
# model.save_pretrained_merged("../../outputs/qwen2_5_7b/unsloth_merged", tokenizer, save_method="merged_16bit")
#
# 导出 GGUF Q4_K_M
# model.save_pretrained_gguf("../../outputs/qwen2_5_7b/unsloth_gguf", tokenizer, quantization_method="q4_k_m")

## 7. 衔接关系

- 只想继续训练或分享小文件：保存 adapter。
- 想给 Transformers/vLLM 使用完整模型：保存 merged model。
- 想给 Ollama/llama.cpp 使用：导出 GGUF。
- 如果训练后模型效果异常，先用 Transformers 或 Unsloth 推理验证，再进入部署。